In [26]:
import sys
from pathlib import Path

ROOT = Path.cwd()

while not (ROOT / "config.py").exists():

    if ROOT.parent == ROOT:
        raise RuntimeError("Project root not found")

    ROOT = ROOT.parent

sys.path.append(str(ROOT))

print("PROJECT ROOT:", ROOT)

PROJECT ROOT: c:\Users\nagal\Documents\AI\rag-benchmark


In [ ]:
CHROMA_PATH = Path(config.CHROMA_PERSIST_DIR)

print("Deleting:", CHROMA_PATH)

if CHROMA_PATH.exists():
    shutil.rmtree(CHROMA_PATH)
    print("✅ Chroma deleted")
else:
    print("ℹ️ Chroma does not exist")

In [ ]:
BM25_INDEX = Path("vectorless_rag/bm25_index.pkl")
BM25_MANIFEST = Path("vectorless_rag/bm25_manifest.json")

for path in [BM25_INDEX, BM25_MANIFEST]:

    if path.exists():
        path.unlink()
        print(f"Deleted {path}")
        

In [ ]:
print("Running preprocessing...")

data = run_preprocessing_pipeline()

print()

print("Parents :", len(data["parents"]))
print("Children:", len(data["children"]))

In [27]:
parent_ids = {
    p["chunk_id"]
    for p in data["parents"]
}

broken = []

for child in data["children"]:

    if child["parent_id"] not in parent_ids:

        broken.append(
            child["chunk_id"]
        )

print(
    "Broken references:",
    len(broken)
)

NameError: name 'data' is not defined

In [ ]:
print("Building vector index...")

collection = index_chunks(data)

print()

print(
    "Vectors in Chroma:",
    collection.count()
)

In [ ]:
print("Building BM25...")

build_bm25_index(data)

print("✅ BM25 rebuilt")

from pathlib import Path

print(Path.cwd())

In [ ]:
from pathlib import Path

BM25_INDEX = (
    ROOT /
    "vectorless_rag" /
    "bm25_index.pkl"
)

BM25_MANIFEST = (
    ROOT /
    "vectorless_rag" /
    "bm25_manifest.json"
)

for file in [BM25_INDEX, BM25_MANIFEST]:

    if file.exists():
        file.unlink()
        print(f"✅ Deleted: {file}")
    else:
        print(f"ℹ️ Not found: {file}")

In [ ]:
print("BM25 Index Exists   :", BM25_INDEX.exists())
print("BM25 Manifest Exists:", BM25_MANIFEST.exists())

In [ ]:
from data_loader import run_preprocessing_pipeline
from vectorless_rag.indexer import build_bm25_index

print("Loading processed data...")

data = run_preprocessing_pipeline()

print(
    f"Parents: {len(data['parents'])}"
)
print(
    f"Children: {len(data['children'])}"
)

print("\nBuilding BM25...")

build_bm25_index(data)

print("✅ BM25 build complete")

In [25]:
from vector_rag.indexer import (
    index_chunks,
    get_chroma_collection
)

collection = get_chroma_collection()

print(
    "Chroma count:",
    collection.count()
)

Chroma count: 20669


In [24]:
results = collection.query(
    query_texts=["microsoft revenue"],
    n_results=3,
    include=[
        "documents",
        "metadatas"
    ]
)

for meta in results["metadatas"][0]:

    print(meta)
    print()

NameError: name 'collection' is not defined

In [4]:
from vector_rag.pipeline import (
    VectorRAGPipeline
)

from vector_rag.retriever import (
    retrieve
)

rag = VectorRAGPipeline()

Loading Vector_pipeline...
AAAYYYOOOOOO
Loading ReRanker...
ReRanker loaded in 29.92s
Vector_pipeline loaded in 34.57s
🔧 Initialising Vector RAG Pipeline...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ ChromaDB loaded — 20669 child vectors
Mistral client ready - model: mistral-medium-latest
✅ Vector RAG ready — 5841 parents in lookup



In [5]:
result = retrieve(
    query="What was NVIDIAs revenue in 2025?",
    collection=rag.collection,
    parent_lookup=rag.parent_lookup,
    top_k=5
)

for i, chunk in enumerate(result["chunks"]):

    print("=" * 80)

    print("Rank:", i + 1)

    print(
        "Metadata Company:",
        chunk["metadata"]["company"]
    )

    parent_id = (
        chunk["metadata"]["parent_id"]
    )

    print(
        "Parent Company:",
        rag.parent_lookup[parent_id]["company"]
    )

    print(
        "Parent ID:",
        parent_id
    )


===== COMPANY MATCH DEBUG =====
QUESTION: what was nvidias revenue in 2025?
MATCHED COMPANY: NVIDIA
RAW: What was NVIDIAs revenue in 2025?
PROCESSED: What was NVIDIAs revenue in 2025?

ORIGINAL : What was NVIDIAs revenue in 2025?
COMPANY  : NVIDIA
YEAR     : 2025
USED FOR SEARCH:
What was NVIDIAs revenue in 2025?

SEARCH QUERY: Represent this sentence for searching relevant passages: What was NVIDIAs revenue in 2025?
🔁 Loading reranker: BAAI/bge-reranker-large


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✅ Reranker ready

========== DEBUG ==========
PARENT ID: parent_4129
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 69
LOOKUP CHUNK: parent_4129

========== DEBUG ==========
PARENT ID: parent_4043
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 53
LOOKUP CHUNK: parent_4043

========== DEBUG ==========
PARENT ID: parent_4172
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 78
LOOKUP CHUNK: parent_4172

========== DEBUG ==========
PARENT ID: parent_4170
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 77
LOOKUP CHUNK: parent_4170

========== DEBUG ==========
PARENT ID: parent_4041
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 52
LOOKUP CHUNK: parent_4041
Rank: 1
Metadata Company: NVIDIA
Parent Company: NVIDIA
Parent ID: parent_4129
Rank: 2
Metadata Company: NVIDIA
Parent Company: NVIDIA
Parent ID: parent_4043
Rank: 3
Metadata Company: NVIDIA
Parent Company: NVIDIA
Parent ID: parent_4172
Rank: 4
Metadata Company: NVIDIA
Parent Company: NV

In [21]:
from pathlib import Path
import json
from tqdm import tqdm

ROOT = Path.cwd().parent

questions_path = (
    ROOT
    / "evaluation"
    / "test_questions_test.json"
)

with open(
    questions_path,
    "r",
    encoding="utf-8"
) as f:

    questions = json.load(f)["questions"]

dataset = []

for item in tqdm(questions):

    result = retrieve(
        query=item["question"],
        collection=rag.collection,
        parent_lookup=rag.parent_lookup,
        top_k=20
    )

    chunks = result["chunks"]

    if not chunks:
        continue

    dataset.append({
        "question": item["question"],
        "positive_chunk": chunks[0]["child_text"],
        "company": item["company"],
        "category": item["category"]
    })

print("Pairs:", len(dataset))

  0%|          | 0/14 [00:00<?, ?it/s]


===== COMPANY MATCH DEBUG =====
QUESTION: what was microsofts total revenue for the fiscal year ended june 30, 2025?
MATCHED COMPANY: MICROSOFT
RAW: What was Microsofts total revenue for the fiscal year ended June 30, 2025?
PROCESSED: What was Microsofts total revenue for the fiscal year ended June 30, 2025?

ORIGINAL : What was Microsofts total revenue for the fiscal year ended June 30, 2025?
COMPANY  : MICROSOFT
YEAR     : 2025
USED FOR SEARCH:
What was Microsofts total revenue for the fiscal year ended June 30, 2025?

SEARCH QUERY: Represent this sentence for searching relevant passages: What was Microsofts total revenue for the fiscal year ended June 30, 2025?
SEARCH QUERY: Represent this sentence for searching relevant passages: What was Microsofts total revenue for the fiscal year ended June 30, 2025?


  7%|▋         | 1/14 [00:19<04:12, 19.45s/it]


========== DEBUG ==========
PARENT ID: parent_2470
CHILD COMPANY: MICROSOFT
LOOKUP COMPANY: MICROSOFT
LOOKUP PAGE: 35
LOOKUP CHUNK: parent_2470

========== DEBUG ==========
PARENT ID: parent_2697
CHILD COMPANY: MICROSOFT
LOOKUP COMPANY: MICROSOFT
LOOKUP PAGE: 85
LOOKUP CHUNK: parent_2697

========== DEBUG ==========
PARENT ID: parent_2693
CHILD COMPANY: MICROSOFT
LOOKUP COMPANY: MICROSOFT
LOOKUP PAGE: 84
LOOKUP CHUNK: parent_2693

========== DEBUG ==========
PARENT ID: parent_2696
CHILD COMPANY: MICROSOFT
LOOKUP COMPANY: MICROSOFT
LOOKUP PAGE: 85
LOOKUP CHUNK: parent_2696

========== DEBUG ==========
PARENT ID: parent_2660
CHILD COMPANY: MICROSOFT
LOOKUP COMPANY: MICROSOFT
LOOKUP PAGE: 76
LOOKUP CHUNK: parent_2660

========== DEBUG ==========
PARENT ID: parent_2637
CHILD COMPANY: MICROSOFT
LOOKUP COMPANY: MICROSOFT
LOOKUP PAGE: 71
LOOKUP CHUNK: parent_2637

========== DEBUG ==========
PARENT ID: parent_2631
CHILD COMPANY: MICROSOFT
LOOKUP COMPANY: MICROSOFT
LOOKUP PAGE: 70
LOOKUP CHUN

 14%|█▍        | 2/14 [00:29<02:44, 13.69s/it]

...


========== DEBUG ==========
PARENT ID: parent_4259
CHILD COMPANY: RELIANCE
LOOKUP COMPANY: RELIANCE
LOOKUP PAGE: 5
LOOKUP CHUNK: parent_4259

========== DEBUG ==========
PARENT ID: parent_4777
CHILD COMPANY: RELIANCE
LOOKUP COMPANY: RELIANCE
LOOKUP PAGE: 49
LOOKUP CHUNK: parent_4777

========== DEBUG ==========
PARENT ID: parent_5635
CHILD COMPANY: RELIANCE
LOOKUP COMPANY: RELIANCE
LOOKUP PAGE: 133
LOOKUP CHUNK: parent_5635

========== DEBUG ==========
PARENT ID: parent_5653
CHILD COMPANY: RELIANCE
LOOKUP COMPANY: RELIANCE
LOOKUP PAGE: 134
LOOKUP CHUNK: parent_5653

========== DEBUG ==========
PARENT ID: parent_5536
CHILD COMPANY: RELIANCE
LOOKUP COMPANY: RELIANCE
LOOKUP PAGE: 124
LOOKUP CHUNK: parent_5536

========== DEBUG ==========
PARENT ID: parent_5358
CHILD COMPANY: RELIANCE
LOOKUP COMPANY: RELIANCE
LOOKUP PAGE: 106
LOOKUP CHUNK: parent_5358

========== DEBUG ==========
PARENT ID: parent_4964
CHILD COMPANY: RELIANCE
LOOKUP COMPANY: RELIANCE
LOOKUP PAGE: 67
LOOKUP CHUNK: parent_4

 93%|█████████▎| 13/14 [02:01<00:08,  8.45s/it]


========== DEBUG ==========
PARENT ID: parent_4129
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 69
LOOKUP CHUNK: parent_4129

========== DEBUG ==========
PARENT ID: parent_3959
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 37
LOOKUP CHUNK: parent_3959

========== DEBUG ==========
PARENT ID: parent_4170
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 77
LOOKUP CHUNK: parent_4170

========== DEBUG ==========
PARENT ID: parent_4172
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 78
LOOKUP CHUNK: parent_4172

========== DEBUG ==========
PARENT ID: parent_4043
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 53
LOOKUP CHUNK: parent_4043

========== DEBUG ==========
PARENT ID: parent_4171
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 77
LOOKUP CHUNK: parent_4171

========== DEBUG ==========
PARENT ID: parent_4022
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 49
LOOKUP CHUNK: parent_4022

========== DEBUG =========

100%|██████████| 14/14 [02:11<00:00,  9.37s/it]


========== DEBUG ==========
PARENT ID: parent_4129
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 69
LOOKUP CHUNK: parent_4129

========== DEBUG ==========
PARENT ID: parent_3959
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 37
LOOKUP CHUNK: parent_3959

========== DEBUG ==========
PARENT ID: parent_4170
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 77
LOOKUP CHUNK: parent_4170

========== DEBUG ==========
PARENT ID: parent_4172
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 78
LOOKUP CHUNK: parent_4172

========== DEBUG ==========
PARENT ID: parent_4043
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 53
LOOKUP CHUNK: parent_4043

========== DEBUG ==========
PARENT ID: parent_4171
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 77
LOOKUP CHUNK: parent_4171

========== DEBUG ==========
PARENT ID: parent_4022
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 49
LOOKUP CHUNK: parent_4022

========== DEBUG =========

In [32]:
import json
from pathlib import Path

output_path = Path(
    "finetuning/training/generated/train_pairs_v2.jsonl"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    for row in dataset:

        f.write(
            json.dumps(
                row,
                ensure_ascii=False
            )
            + "\n"
        )

print(
    f"Saved {len(dataset)} pairs to:"
)
print(output_path)

Saved 14 pairs to:
finetuning\training\generated\train_pairs_v2.jsonl


In [31]:
import json

path = (
    "finetuning/training/generated/"
    "train_pairs_v2.jsonl"
)

bad = []

with open(path, encoding="utf-8") as f:

    for line in f:

        row = json.loads(line)

        expected = (
            row["company"]
            .upper()
            .replace("-", "")
            .replace(" ", "")
        )

        retrieved = (
            row["retrieved_company"]
            .upper()
            .replace("-", "")
            .replace(" ", "")
        )

        if expected != retrieved:

            bad.append(row)

print(
    "Total pairs:",
    len(open(path).readlines())
)

print(
    "Mismatched company pairs:",
    len(bad)
)

for x in bad[:10]:
    print()
    print("QUESTION:", x["question"])
    print("EXPECTED:", x["company"])
    print("RETRIEVED:", x["retrieved_company"])

KeyError: 'retrieved_company'